# Validation Notebook for SSLimPy halomodel 

In this notebook we will study the different components of the LIM halo model computation and compare them to the Colossus results

## Initial Setup

In [ ]:
import os
import sys

sys.path.append("../")

envkey = "OMP_NUM_THREADS"
# Set this environment variable to the number of available cores in your machine,
# to get a fast execution of the Einstein Boltzmann Solver
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))
os.environ[envkey] = str(12)
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
from copy import copy, deepcopy
import numpy as np
import seaborn
from getdist.gaussian_mixtures import GaussianND
from getdist import plots
import matplotlib.patches as mpatches

In [ ]:
# seaborn.set_theme(rc={'axes.edgecolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black',})
import niceplots.utils as nicepl
nicepl.initPlot()

In [ ]:
from scipy.signal import find_peaks
from scipy.interpolate import UnivariateSpline

In [ ]:
Cs = seaborn.color_palette("Paired")
Cs

In [ ]:
Cp = seaborn.color_palette("colorblind")
Cp

## Choose model parameters and save them in dictionaries

In [ ]:
settings = {
    "code":"class",
    "do_RSD" : False,
    "nonlinearRSD" : True,
    "QNLpowerspectrum": False,
    "FoG_damp" : "ISTF_like",
    "halo_model_PS" : True,  
    "output" : ["Power spectrum", "Covariance"],
    "FFTlog_LogN" : 13,
}

cosmodict={
    "h": 0.6737,
    "Omegam": 0.3146,
    "Omegab": 0.0492,
    "sigma8":0.8,
    "ns":0.966,
    "mnu":0.06,
    "Neff":3.044,
}

halodict={
    "halo_tracer" : "matter", 
    "hmf_model": "ST", 
    "concentration": "Diemer19",
    "bias_model": "b1",
    "bias_pars": {
        "HO_p" : 0.3,
        "HO_alpha" : 0.707,
        "HO_A" : 0.3222,
    },
    "transition_smoothing": True,
    "bloating": False,
    "onehalo_damping": True,
}

astrodict={
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {
        "a": 0.939,
        "b": 6.639,
        "SFR_file": "Fonseca16_Lya_SFR_params.dat",
        "do_quench": True,
    },
    "sigma_scatter" : 0.5
}

rho_m = cosmodict["Omegam"] * cosmodict["h"]**2 * 2.77536627e11 #Msun / Mpc**3

astrodict2 = {
    "model_type": "ML",
    "model_name": "MassPow",
    "model_par": {
        "A": 1 / rho_m,
        "b": 1.0, # This models the M / rho scaling of the halo--matter power spectrum
    },
    "sigma_scatter" : 0.0
}

# Parameters for the Survey specifications
surveyspecs = {
        "Tsys_NEFD": 40 * u.uK, #System temperature for instrumental shotnoise
        "Nfeeds": 19,
        "tobs": 1300 * u.h,
        "nD": 1, # Observational parameters
        "beam_FWHM": 0.5 * u.arcmin,
        "nu":  1.897 * u.THz, # CII
        "dnu": 380 * u.MHz, # Spectrograph resolution
        "nuObs": 950 * u.GHz, # Observed Frequency
        "Delta_nu": 50 * u.GHz, # Frequency Bin
        "Omega_field": 140 * u.deg**2, # Angular size of survey
}

## Compute instance of SSLimPy to work with

In [ ]:
# Import Main modules. This might take some time as some functions compile before time
from SSLimPy.interface import sslimpy
from SSLimPy.interface import updater
from SSLimPy.cosmology import cosmology
from SSLimPy.cosmology import halo_model
from SSLimPy.cosmology import astro
from SSLimPy.LIMsurvey import power_spectrum
from SSLimPy.LIMsurvey import covariance
from SSLimPy.LIMsurvey import higher_order
from SSLimPy.LIMsurvey import ingredients_T0

In [ ]:
from SSLimPy.utils.utils import *

In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
    astropars=astrodict,
    obspars_dict=surveyspecs,
)

In [ ]:
cosmo = myssl.current_cosmology
halo = myssl.current_halomodel
myyastro = myssl.current_astro

In [ ]:
from colossus.cosmology import cosmology
from colossus.lss import mass_function
from colossus.lss import bias
from colossus.halo import mass_so
from colossus.halo import concentration

In [ ]:
Colossus_cosmo = cosmology.setCosmology(
    "mycosmo",
    {
        "flat":True,
        "H0": cosmodict["h"] * 100,
        "Ob0": cosmodict["Omegab"],
        "sigma8": cosmodict["sigma8"],
        "ns": cosmodict["ns"],
        "Om0": cosmodict["Omegam"],
    }
)

In [ ]:
# all lenghts and masses in Colossus are rescaled by h
h = cosmo.h()


In [ ]:
z = np.linspace(0, 4)
cDa = Colossus_cosmo.angularDiameterDistance(z) / h
sDa = cosmo.angdist(z)

plt.semilogy(z, cDa, c=Cs[0], label="Colossus")
plt.semilogy(z, sDa, ls="--", c=Cs[1], label="SSLimPy")
plt.legend()
plt.xlabel("redshift $z$")
plt.ylabel(r"$D_\mathrm{A}\,[\mathrm{Mpc}]$")
plt.tight_layout()

In [ ]:
k = np.geomspace(1e-3, 1, 50) * u.Mpc**-1
cPk = Colossus_cosmo.matterPowerSpectrum(k.value / h, 0) / h**3
sPk = cosmo.matpow(k, 0)

plt.loglog(k, cPk, c=Cs[0], label="Colossus")
plt.loglog(k, sPk, ls="--", c=Cs[1], label="SSLimPy")
plt.legend()
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P(k)\,[\mathrm{Mpc}^3]$")
plt.tight_layout()

In [ ]:
R = np.geomspace(1, 100, 20) * u.Mpc
csR = Colossus_cosmo.sigma(R.value * h, 1.0)
ssR = halo.sigmaR_of_z(R, 1.0)

fig, axs = plt.subplots(2, 1, sharex=True, gridspec_kw={"height_ratios": [5, 2]})
axs[0].loglog(R, csR, c=Cs[0], label="Colossus")
axs[0].loglog(R, ssR, ls="--", c=Cs[1], label="SSLimPy")
axs[1].semilogx(R, (csR / ssR - 1) * 100, c=Cs[1])

axs[0].legend()
axs[1].set_xlabel(r"$R\,[\mathrm{Mpc}]$")
axs[0].set_ylabel(r"$\sigma_R$")

plt.tight_layout()
fig.subplots_adjust(hspace=0.0)

In [ ]:
M = np.geomspace(1e9, 10**13.5) * halo.Msunh
M = M.to(halo.Msunh)

In [ ]:
cnu = mass_function.peaks.peakHeight(M.value, 1.0)
snu = halo.delta_crit / halo.sigmaM(M, 1.0, halo.tracer)

plt.semilogx(M, cnu, c=Cs[0], label="Collosus")
plt.semilogx(M, snu, c=Cs[1], label="SSLimPy", ls="--")
plt.legend()
plt.xlabel(r"$M\,[h^{-1}\,M_\odot]$")
plt.ylabel(r"collapse fraction $\nu$")

In [ ]:
cmf=mass_function.massFunction(M.value, 1.0, q_in="M", q_out="dndlnM", model="sheth99")
smf=(halo.halomassfunction(M, 1.0) * M).to(halo.Mpch**-3)

fig, axs = plt.subplots(2, 1, sharex=True, gridspec_kw={"height_ratios": [5, 2]})

axs[0].loglog(M, cmf, c=Cs[0], label="Colossus")
axs[0].loglog(M, smf, ls="--", c=Cs[1], label="SSLimPy")
axs[1].semilogx(M, (smf.value/cmf -1) * 100, c=Cs[1])

axs[1].set_xlabel(r"$M\,[h^{-1}\,M_\odot]$")
axs[0].set_ylabel(r"$\mathrm{d}N\,/\,\mathrm{dlog}M\,[\mathrm{Mpc}^{-3}]$")
axs[0].legend()

plt.tight_layout()
fig.subplots_adjust(hspace=0.0)

In [ ]:
cb1 = bias.haloBias(M.value, 1.0, model="sheth01")
sb1 = halo.get_bias(M, 1.0, beta=1)
plt.loglog(M, cb1, c=Cs[0], label="Colossus")
plt.loglog(M, sb1, ls="--", c=Cs[1], label="SSLimPy")
plt.legend()
plt.xlabel(r"$M\,[h^{-1}\,M_\odot]$")
plt.ylabel(r"halo bias $b_1$")

In [ ]:
M = np.geomspace(1e10, 1e15) * u.Msun
ccM = concentration.concentration(M.value * h, "200c", 0.0, "diemer19", "mean")
scM = halo.concentration(M, 0.0)

plt.semilogx(M, ccM, c=Cs[0], label="Colossus")
plt.semilogx(M, scM, c=Cs[1], label="SSLimPy", ls="--")
plt.legend()
plt.xlabel(r"$M\,[M_\odot]$")

In [ ]:
k = np.geomspace(1e-3, 10, 100) * u.Mpc**-1
I0 = halo.Ihalo(1.0, k, p=1, scale=(2,), beta="b0")
I1 = halo.Ihalo(1.0, k, p=1, scale=(1,), beta="b1")
plin = cosmo.matpow(k, 1.0, nonlinear=False, tracer=halo.tracer)
phalo = I1**2 * plin + I0
pnl = cosmo.matpow(k, 1.0, nonlinear=True, tracer=halo.tracer)

plt.loglog(k, plin, "k--")
plt.loglog(k, pnl, c=Cs[3], label="HMCode")
plt.loglog(k, phalo, c=Cs[0], label="SSLimPy (base)")
plt.loglog(k, halo.P_halo(k, 1.0), c=Cs[1], label="SSLimPy (fitted)")
plt.legend()
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P(k)\,[\mathrm{Mpc}^3]$")